# Part 1: Dataset Exploration and Preparation

In this module, we will explore the Odia character dataset, verify its structure, and filter out any invalid classes to ensure our dataset is robust for machine learning.

In [ ]:
import os
import cv2
import shutil
import random
import datetime
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
def load_root_env(env_name='.env'):
    """Load simple KEY=VALUE entries from the repo root .env without overwriting existing env vars."""
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        env_path = folder / env_name
        if env_path.is_file():
            for line in env_path.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, value = line.split('=', 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
            print('Loaded environment from:', env_path)
            return env_path
    print('No root .env found; using existing environment only.')
    return None

load_root_env()

# Local dataset/complete_dataset is the default working copy.
# It can be restored from Hugging Face with scripts/dataset/download_hf.py.
PROJECT_ROOT = Path(os.environ.get("LIPY_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "training":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = Path(
    os.environ.get("LIPY_DATASET_DIR", PROJECT_ROOT / "dataset" / "complete_dataset")
).resolve()

print("Dataset Path:", DATASET_DIR)

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATASET_DIR}. "
        "Run `python scripts/setup.py` from the repo root first, "
        "or set LIPY_DATASET_DIR to a local folder."
    )

files = [f.name for f in DATASET_DIR.iterdir() if f.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
print("Total Images Found:", len(files))
if files:
    print("Sample Images:", files[:5])


In [ ]:
from collections import Counter

# Count samples for each class directly from the flat folder filenames.
class_counts = Counter()
for image_path in DATASET_DIR.iterdir():
    if image_path.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
        continue
    parts = image_path.name.split('_')
    if len(parts) >= 2:
        class_name = f"{parts[0]}_{parts[1]}"
        class_counts[class_name] += 1

# Filter classes with a minimum threshold of images to ensure training stability.
MIN_IMAGES = 25
valid_classes = [class_name for class_name, count in class_counts.items() if count >= MIN_IMAGES]

print(f"{len(valid_classes)} classes meet the threshold of {MIN_IMAGES} images.")

In [ ]:
valid_classes.sort()
label_map = {class_name: idx for idx, class_name in enumerate(valid_classes)}
print("Label map constructed successfully.")